# SportsMOT – step 2: intra-game hard-negative Re-ID for OAF-Track
Trains an appearance model on player crops from the SportsMOT **train** split
(val and test are never used for training).

**Key idea – intra-game hard-negative sampling:** every training batch draws its players from only
two matches, so most negatives in the triplet loss are *teammates / opponents of the same game*
(same kit, same lighting). This forces the network to learn the fine cues (jersey number, body shape,
shoes, hair) that separate players who look alike – exactly the case where ImageNet features fail.

Model: ResNet-50 (ImageNet init, last stride 1) + BNNeck; loss = label-smoothed ID loss + batch-hard
triplet loss (Luo et al., "Bag of Tricks and a Strong Baseline for Deep Person Re-ID", 2019).

Run the cells top to bottom. The model is saved to `WORK_DIR/reid/sportsreid_r50.pth`.

In [ ]:
# ── Cell 1: settings ────────────────────────────────────────────────────
import os, sys, time, math, random, configparser
from pathlib import Path
import numpy as np, cv2
from tqdm.auto import tqdm

DATA_ROOT = Path(r"data/SportsMOT")
WORK_DIR  = Path(r"outputs/sportsmot")
REID_DIR  = WORK_DIR / 'reid'

TRAIN_STRIDE = 10       # take every 10th frame of each train video
MAX_PER_ID   = 80       # at most this many crops per player identity
MIN_H        = 40       # skip boxes smaller than this (pixels)
IMG_H, IMG_W = 256, 128

EPOCHS       = 40
SEQ_PER_B    = 2        # matches per batch  (intra-game hard negatives)
P, K         = 16, 4    # identities per batch, images per identity -> batch 64
BASE_LR      = 3.5e-4
SEED         = 0

random.seed(SEED); np.random.seed(SEED)
REID_DIR.mkdir(parents=True, exist_ok=True)
print('Re-ID work folder:', REID_DIR)

In [ ]:
# ── Cell 2: extract player crops (train = training, val = evaluation only) ──
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def is_seq(d):
    return d.is_dir() and (d / 'img1').is_dir()

def find_split(root, split):
    for c in [root / split, root / 'dataset' / split, root / 'sportsmot_publish' / 'dataset' / split,
              root / 'dancetrack' / split]:
        if c.is_dir() and any(is_seq(d) for d in c.iterdir()):
            return c
    raise FileNotFoundError(f'"{split}" split not found under {root}')

def extract_crops(split_dir, out_dir, stride, max_per_id):
    """Save GT player crops as <out>/<seq>/<pid>_<frame>.jpg ; returns list of (path, seq, pid, frame)."""
    rows = []
    for s in tqdm(sorted(d for d in split_dir.iterdir() if is_seq(d)), desc=f'crops {split_dir.name}'):
        gt_file = s / 'gt' / 'gt.txt'
        if not gt_file.exists():
            continue
        gt = np.loadtxt(gt_file, delimiter=',', ndmin=2)
        if gt.shape[1] >= 7:
            gt = gt[gt[:, 6] != 0]
        frames = sorted(p for p in (s / 'img1').iterdir() if p.suffix.lower() in IMG_EXTS)
        fmap = {int(p.stem): p for p in frames}
        keep_f = set(sorted(fmap)[::stride])
        count = {}
        (out_dir / s.name).mkdir(parents=True, exist_ok=True)
        for f in sorted(keep_f):
            r = gt[gt[:, 0] == f]
            if not len(r):
                continue
            img = None
            for _, pid, x, y, w, h, *_ in r:
                pid = int(pid)
                if h < MIN_H or count.get(pid, 0) >= max_per_id:
                    continue
                dst = out_dir / s.name / f'{pid:04d}_{int(f):06d}.jpg'
                if not dst.exists():
                    if img is None:
                        img = cv2.imread(str(fmap[f]))
                    H, W = img.shape[:2]
                    x1, y1 = int(max(0, x)), int(max(0, y))
                    x2, y2 = int(min(W, x + w)), int(min(H, y + h))
                    if x2 - x1 < 8 or y2 - y1 < 16:
                        continue
                    cv2.imwrite(str(dst), cv2.resize(img[y1:y2, x1:x2], (IMG_W, IMG_H)))
                count[pid] = count.get(pid, 0) + 1
                rows.append((str(dst), s.name, pid, int(f)))
    return rows

def load_or_extract(split, stride, max_per_id):
    index = REID_DIR / f'crops_{split}.csv'
    if index.exists():
        rows = [l.rstrip('\n').split(',') for l in open(index)]
        crop_root = REID_DIR / 'crops' / split
        out = []
        for p, s, i, f in rows:
            if not os.path.exists(p):                     # work folder was moved: rebuild the path
                p = str(crop_root / s / Path(p.replace('\\', '/')).name)
            out.append((p, s, int(i), int(f)))
        missing = sum(not os.path.exists(p) for p, *_ in out)
        if missing:
            print(f'WARNING: {missing} {split} crops not found – delete {index.name} to re-extract')
        return out
    rows = extract_crops(find_split(DATA_ROOT, split), REID_DIR / 'crops' / split, stride, max_per_id)
    with open(index, 'w') as fh:
        for r in rows:
            fh.write(','.join(map(str, r)) + '\n')
    return rows

TRAIN_ROWS = load_or_extract('train', TRAIN_STRIDE, MAX_PER_ID)
VAL_ROWS   = load_or_extract('val', 25, 20)          # small evaluation set
gid = {}
for _, s, pid, _ in TRAIN_ROWS:
    gid.setdefault((s, pid), len(gid))
NUM_IDS = len(gid)
print(f'train crops: {len(TRAIN_ROWS)} | identities: {NUM_IDS} | matches: {len({r[1] for r in TRAIN_ROWS})}')
print(f'val crops  : {len(VAL_ROWS)}')

In [ ]:
# ── Cell 3: model, losses, intra-game PK sampler ───────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEV, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'NO GPU – training will be slow')

class SportsReID(nn.Module):
    """ResNet-50 (last stride 1) + BNNeck. forward() returns (logits, feat) in train mode,
    the L2-normalised BN feature in eval mode."""
    def __init__(self, num_classes=0, pretrained=True):
        super().__init__()
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        r = torchvision.models.resnet50(weights=w)
        r.layer4[0].conv2.stride = (1, 1)
        r.layer4[0].downsample[0].stride = (1, 1)
        self.backbone = nn.Sequential(*list(r.children())[:-2])
        self.bnneck = nn.BatchNorm1d(2048)
        self.bnneck.bias.requires_grad_(False)
        self.classifier = nn.Linear(2048, num_classes, bias=False) if num_classes else None
    def forward(self, x):
        f = F.adaptive_avg_pool2d(self.backbone(x), 1).flatten(1)
        fb = self.bnneck(f)
        if self.training:
            return self.classifier(fb), f
        return F.normalize(fb, dim=1)

def batch_hard_triplet(feat, labels, margin=0.3):
    d = torch.cdist(feat, feat)
    same = labels[:, None] == labels[None, :]
    hardest_pos = (d * same.float()).max(1).values
    hardest_neg = (d + same.float() * 1e6).min(1).values
    return F.relu(hardest_pos - hardest_neg + margin).mean()

class IntraGamePKSampler:
    """Each batch: SEQ_PER_B matches, P/SEQ_PER_B identities from each, K crops per identity."""
    def __init__(self, rows, n_batches):
        self.by_seq = {}
        for i, (_, s, pid, _) in enumerate(rows):
            self.by_seq.setdefault(s, {}).setdefault(pid, []).append(i)
        self.by_seq = {s: {p: ix for p, ix in d.items() if len(ix) >= 2}
                       for s, d in self.by_seq.items()}
        self.by_seq = {s: d for s, d in self.by_seq.items() if len(d) >= 2}
        self.seqs = list(self.by_seq)
        self.n = n_batches
    def __iter__(self):
        per_seq = P // SEQ_PER_B
        for _ in range(self.n):
            batch = []
            for s in random.sample(self.seqs, min(SEQ_PER_B, len(self.seqs))):
                ids = list(self.by_seq[s])
                for pid in random.sample(ids, min(per_seq, len(ids))):
                    ix = self.by_seq[s][pid]
                    batch += random.sample(ix, K) if len(ix) >= K else random.choices(ix, k=K)
            yield batch
    def __len__(self):
        return self.n

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

def load_img(path, train):
    img = cv2.imread(path)[:, :, ::-1]
    if img.shape[:2] != (IMG_H, IMG_W):
        img = cv2.resize(img, (IMG_W, IMG_H))
    if train:
        if random.random() < 0.5:
            img = img[:, ::-1]
        img = cv2.copyMakeBorder(np.ascontiguousarray(img), 10, 10, 10, 10, cv2.BORDER_CONSTANT)
        y, x = random.randint(0, 20), random.randint(0, 20)
        img = img[y:y + IMG_H, x:x + IMG_W]
    x = (img.astype(np.float32) / 255. - MEAN) / STD
    if train and random.random() < 0.5:                      # random erasing
        eh, ew = random.randint(20, IMG_H // 2), random.randint(10, IMG_W // 2)
        ey, ex = random.randint(0, IMG_H - eh), random.randint(0, IMG_W - ew)
        x[ey:ey + eh, ex:ex + ew] = np.random.randn(eh, ew, 3)
    return x.transpose(2, 0, 1)

@torch.no_grad()
def embed(model, paths, bs=128):
    model.eval(); out = []
    for i in range(0, len(paths), bs):
        x = torch.from_numpy(np.stack([load_img(p, False) for p in paths[i:i + bs]])).to(DEV)
        with torch.autocast(device_type='cuda', enabled=(DEV == 'cuda')):
            out.append(model(x).float().cpu())
    return torch.cat(out).numpy()

def evaluate_within_game(feats, rows):
    """Retrieval among players of the SAME match: query = later 30 % of each track, gallery = earlier 70 %."""
    aps, r1 = [], []
    seqs = {}
    for i, (_, s, pid, f) in enumerate(rows):
        seqs.setdefault(s, []).append((i, pid, f))
    for s, items in seqs.items():
        q, g = [], []
        by_id = {}
        for i, pid, f in items:
            by_id.setdefault(pid, []).append((f, i))
        for pid, lst in by_id.items():
            lst.sort(); cut = max(1, int(len(lst) * 0.7))
            if len(lst) < 2:
                continue
            g += [(i, pid) for _, i in lst[:cut]]; q += [(i, pid) for _, i in lst[cut:]]
        if not q or not g:
            continue
        G = feats[[i for i, _ in g]]; gl = np.array([p for _, p in g])
        for i, pid in q:
            sim = G @ feats[i]; order = np.argsort(-sim); match = gl[order] == pid
            if not match.any():
                continue
            r1.append(match[0])
            hits = np.cumsum(match); prec = hits / np.arange(1, len(match) + 1)
            aps.append((prec * match).sum() / match.sum())
    return 100 * float(np.mean(aps)), 100 * float(np.mean(r1))

print('ready')

In [ ]:
# ── Cell 4: baseline – how well do ImageNet features separate players of the same match? ──
base = SportsReID(0).to(DEV)
vf = embed(base, [r[0] for r in VAL_ROWS])
vf = vf - vf.mean(0); vf /= np.linalg.norm(vf, axis=1, keepdims=True) + 1e-12
BASE_MAP, BASE_R1 = evaluate_within_game(vf, VAL_ROWS)
print(f'ImageNet ResNet-50 (no training): within-game mAP = {BASE_MAP:.1f}  Rank-1 = {BASE_R1:.1f}')
del base

In [ ]:
# ── Cell 5: train (resumes automatically if interrupted) ───────────────
labels_all = np.array([gid[(s, pid)] for _, s, pid, _ in TRAIN_ROWS])
paths_all = [r[0] for r in TRAIN_ROWS]
iters = max(1, len(TRAIN_ROWS) // (P * K))
model = SportsReID(NUM_IDS).to(DEV)
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=BASE_LR, weight_decay=5e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(DEV == 'cuda'))
ce = nn.CrossEntropyLoss(label_smoothing=0.1)
ckpt = REID_DIR / 'train_state.pth'
start = 0
if ckpt.exists():
    try:
        st = torch.load(ckpt, map_location=DEV, weights_only=False)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); start = st['epoch'] + 1
        print('Resuming from epoch', start)
    except Exception as e:                                # e.g. half-written file after "disk full"
        print(f'Checkpoint unreadable ({type(e).__name__}) – starting training from epoch 1')
        model = SportsReID(NUM_IDS).to(DEV)
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                               lr=BASE_LR, weight_decay=5e-4)

def lr_at(ep):
    if ep < 10:
        return BASE_LR * (0.1 + 0.9 * ep / 10)               # warm-up
    return BASE_LR * (0.1 if ep >= int(EPOCHS * 0.75) else (0.5 if ep >= int(EPOCHS * 0.5) else 1.0))

for ep in range(start, EPOCHS):
    for g in opt.param_groups:
        g['lr'] = lr_at(ep)
    model.train(); t0 = time.time(); tot = 0.0
    for batch in tqdm(IntraGamePKSampler(TRAIN_ROWS, iters), desc=f'epoch {ep + 1}/{EPOCHS}', leave=False):
        x = torch.from_numpy(np.stack([load_img(paths_all[i], True) for i in batch])).to(DEV)
        y = torch.from_numpy(labels_all[batch]).to(DEV)
        with torch.autocast(device_type='cuda', enabled=(DEV == 'cuda')):
            logits, feat = model(x)
            loss = ce(logits.float(), y) + batch_hard_triplet(feat.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()
    tmp_ck = ckpt.with_suffix('.tmp')                   # write to a temp file, then swap: never corrupt
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'epoch': ep}, tmp_ck)
    os.replace(tmp_ck, ckpt)
    print(f'epoch {ep + 1:2d}/{EPOCHS}  loss {tot / iters:.3f}  lr {lr_at(ep):.2e}  {time.time() - t0:.0f}s')

In [ ]:
# ── Cell 6: evaluate on val + save the final Re-ID model ───────────────
vf = embed(model, [r[0] for r in VAL_ROWS])
vf = vf - vf.mean(0); vf /= np.linalg.norm(vf, axis=1, keepdims=True) + 1e-12
NEW_MAP, NEW_R1 = evaluate_within_game(vf, VAL_ROWS)
print('Within-game retrieval on SportsMOT val (players of the same match):')
print(f'  ImageNet ResNet-50     mAP {BASE_MAP:5.1f}   Rank-1 {BASE_R1:5.1f}')
print(f'  SportsReID (ours)      mAP {NEW_MAP:5.1f}   Rank-1 {NEW_R1:5.1f}')
state = {k: v for k, v in model.state_dict().items() if not k.startswith('classifier')}
final = REID_DIR / 'sportsreid_r50.pth'
torch.save({'state_dict': state, 'img_size': (IMG_H, IMG_W), 'epochs': EPOCHS,
            'val_map': NEW_MAP, 'val_rank1': NEW_R1, 'base_map': BASE_MAP, 'base_rank1': BASE_R1}, final)
with open(REID_DIR / 'reid_results.txt', 'w') as fh:
    fh.write(f'ImageNet ResNet-50: mAP {BASE_MAP:.2f} Rank-1 {BASE_R1:.2f}\n')
    fh.write(f'SportsReID (ours) : mAP {NEW_MAP:.2f} Rank-1 {NEW_R1:.2f}\n')
print('Saved:', final)
print('Next: in the main notebook set  REID_BACKBONE = "sportsreid"  and run all cells.')